In [2]:
import numpy as np
import mcstasscript as ms

In [ ]:
KVASIR = ms.McStas_instr("KVASIR")
#EXAMPLE: interated intensity on detector when running vanadium as sample: 0.0268478
t  = KVASIR.add_parameter("t", value=0) # Wedge no. (330 total to cover 0-170deg)

l_analyzers =[2.5035035035035036, 2.5105105105105103, 2.5175175175175175, 2.5245245245245247, 2.5305305305305303, 2.5375375375375375, 2.5435435435435436, 2.5505505505505504, 2.5565565565565564, 2.5625625625625625, 2.5695695695695697, 2.5755755755755754, 2.5815815815815815, 2.5875875875875876, 2.5925925925925926, 2.5985985985985987, 2.6046046046046047, 2.6096096096096097, 2.6156156156156154, 2.6206206206206204, 2.6266266266266265, 2.6316316316316315, 2.6366366366366365, 2.6416416416416415, 2.6466466466466465, 2.6516516516516515, 2.6566566566566565, 2.6616616616616615, 2.6656656656656654, 2.6706706706706704, 2.674674674674675, 2.67967967967968, 2.6836836836836837, 2.6886886886886887, 2.6926926926926926, 2.6966966966966965, 2.7007007007007005, 2.704704704704705, 2.7087087087087087, 2.7127127127127126, 2.7157157157157155, 2.71971971971972, 2.7237237237237237, 2.7267267267267266, 2.7307307307307305, 2.7337337337337337, 2.736736736736737, 2.7407407407407405, 2.7437437437437437, 2.746746746746747, 2.74974974974975, 2.7527527527527527, 2.754754754754755, 2.7577577577577577, 2.7607607607607605, 2.7637637637637638, 2.7657657657657655, 2.7687687687687688, 2.7707707707707705, 2.7727727727727727, 2.7757757757757755, 2.7777777777777777, 2.77977977977978, 2.781781781781782, 2.7837837837837838, 2.7857857857857855, 2.7877877877877877, 2.78978978978979, 2.7907907907907905, 2.7927927927927927, 2.793793793793794, 2.7957957957957955, 2.796796796796797, 2.798798798798799, 2.7997997997998, 2.8008008008008005, 2.801801801801802, 2.8028028028028027, 2.804804804804805, 2.804804804804805] #array containg nummerical solution for distance from sample to n'th analyzer in wedge

N_analyzer =60                                                      #1 total number of analyzers
width = 0.0127                                                      #m width of He3 tubes, height of analyzer crystals
dt = 2*np.arcsin(width/(2*2.5))*180/np.pi                           #deg vertical angle between each analyzer crystal seen from sample position
theta =  np.linspace(-dt/2,-dt*N_analyzer-dt/2, N_analyzer)         #deg array containg rotations

In [4]:
# PARAMETERS
width= KVASIR.add_declare_var("double", "width", value=0.0127)                              #m width of He3 tubes, height of analyzer crystals
d_PG= KVASIR.add_declare_var("double", "d_PG", value=3.355)                                 #Å lattice spacing og pyrolytic graphite
u = KVASIR.add_declare_var("double","u", value=1e-5)                                        #1 small number
h_D = KVASIR.add_declare_var("double", "h_D", value=0.4)                                    #m heigt of detector 

Ef = KVASIR.add_declare_var("double","Ef", value=1.95)                                      #meV focusing energy                       
d_AD = KVASIR.add_declare_var("double","d_AD", value=1.25)                                  #m distance from analyzer to detactor
d_SA = KVASIR.add_declare_var("double","d_SA", value=2.5)                                   #m distance from sample to central analyzer
d_GS = KVASIR.add_declare_var("double","d_GS", value=0.58)                                  #m distance from guide end to sample position

lamda_f = KVASIR.add_declare_var("double", "lambda_f")                                      #Å final wavelenght
KVASIR.append_initialize("lambda_f=1/(0.11056*sqrt(Ef));")

#ANALYZER CALCULATION
tA = KVASIR.add_declare_var("double", "tA")                                                 #deg Bragg angle of analyzer
KVASIR.append_initialize("tA=asin(lambda_f/(2*d_PG))*RAD2DEG;")

dtS = KVASIR.add_declare_var("double", "dtS", value=[*theta], array = len(theta))           #deg array containg rotations

len = KVASIR.add_declare_var("double", "len", value=l_analyzers, array = len(l_analyzers))  #array containg nummerical solution for distance from sample to n'th analyzer in wedge

#DETECTOR CALCULATION
d_SDz = KVASIR.add_declare_var("double", "d_SDz")                                           #m vetical distance from scattering plane to center of detector
KVASIR.append_initialize("d_SDz=d_SA+d_AD*cos(2*tA*DEG2RAD);")

d_SDy = KVASIR.add_declare_var("double", "d_SDy")                                           #m distance from sample to detector in scattering plane
KVASIR.append_initialize("d_SDy=d_AD*sin(2*tA*DEG2RAD);")

tDy= KVASIR.add_declare_var("double", "tDy")                                                #deg horizontal angle between wedges                                          
KVASIR.append_initialize("tDy=asin(width/(d_SDz))*RAD2DEG;")

In [ ]:
## SOURCE OPTION 1: DUMMY SOURCE OPTION
# source = KVASIR.add_component(f"source", "Source_Maxwell_3", AT=[0,0,0], RELATIVE="ABSOLUTE")
# source.xwidth=0.01
# source.yheight=0.01
# source.Lmin= "lambda_f-D_lambda/2"
# source.Lmax= "lambda_f+D_lambda/2"
# source.dist= d_SA
# source.focus_yh= f"width*2*{N_analyzer}"
# source.focus_xw=f"width*3"
# source.T1=300
# source.T2=300
# source.T3=300
# source.I1=1E15
# source.I2=1E15
# source.I3=1E15

## SOURCE OPTION 2: IMPORT MCPL FILE FROM PRIMARY SPECTROMETER
source = KVASIR.add_component(f"source", "MCPL_input", AT=[0,0,0], RELATIVE="ABSOLUTE")
source.filename= '"Virtual_output_endOfGuide_narrow_100mus.mcpl"' #EXAMPLE MCPL: BIFROST GUIDE, 100mus PSC opening time, wavelength band is centered at 1.95meV and narrowed to speed up simulations
source.pos_smear = 0.005
source.dir_smear = 0.05

#ARM DEFINES SAMPLE POSITION (0.58m is distance from BIFROST guide opening to sample position)
Arm_sample = KVASIR.add_component(f"Arm_sample", "Arm", AT=[0,0,d_GS], ROTATED = [0,f"tDy*t",0], RELATIVE="ABSOLUTE")

## SAMPLE OPTION 1: VANADIUM
sample = KVASIR.add_component(f"sample", "Incoherent", AT=[0,0,0], RELATIVE="Arm_sample")
sample.radius=0.005
sample.yheight=0.01
sample.thickness=0.001
sample.target_index = 2
sample.focus_ah =0.5
sample.focus_aw = 20 

## SAMPLE OPTION 1: POWDER SAMPLE
# sample = KVASIR.add_component(f"sample", "PowderN", AT=[0,0,0], RELATIVE="Arm_sample")
# sample.radius=0.005
# sample.yheight=0.01
# sample.thickness=0.001
# sample.reflections="Na2Ca3Al2F14.laz"

In [6]:
for n in range(0,N_analyzer,2):                                                                  #loops placing analyzers (every second analyzer is placed to avoid shading)

    az = KVASIR.add_declare_var("double", f"az_{n}")                                             #m z-component of n'th analyzer relative to sample position
    KVASIR.append_initialize(f"az_{n} = len[{n}]*cos(dtS[{n}]*DEG2RAD)-d_SDz;")
    
    ay = KVASIR.add_declare_var("double", f"ay_{n}")                                             #m y-component of n'th analyzer relative to sample position
    KVASIR.append_initialize(f"ay_{n} = len[{n}]*sin(dtS[{n}]*DEG2RAD)-d_SDy;")

    bz = KVASIR.add_declare_var("double", f"bz_{n}")                                             #m z-component of detector relative to n'th analyzer
    KVASIR.append_initialize(f"bz_{n} = len[{n}]*cos(dtS[{n}]*DEG2RAD);")                        
    
    by = KVASIR.add_declare_var("double", f"by_{n}")                                             #m y-component of detector relative to n'th analyzer
    KVASIR.append_initialize(f"by_{n} = len[{n}]*sin(dtS[{n}]*DEG2RAD);")
    
    dtA = KVASIR.add_declare_var("double", f"dtA_{n}")                                           #deg scattering angle of n'th analyzer
    KVASIR.append_initialize(f"dtA_{n} = acos( (az_{n}*bz_{n} + ay_{n}*by_{n}) / (len[{n}]* sqrt(az_{n}*az_{n} + ay_{n}*ay_{n}) ) )*RAD2DEG;")

    # GENERARING UPPER ANALYZER WEDGE
    Arm = KVASIR.add_component(f"Arm_{n}", "Arm", AT=[0,0,0], ROTATED=[f"-dtS[{n}]",0,0], RELATIVE="Arm_sample") 
    Analyser = KVASIR.add_component(f"Analyzer_{n}","Monochromator_flat", AT = [0,0,f"len[{n}]"], ROTATED=[f"dtA_{n}/2",90,0], RELATIVE=f"Arm_{n}")
    Analyser.zwidth = f"2*len[{n}]*sin(tDy/2*DEG2RAD)"
    Analyser.yheight = width
    Analyser.mosaich = 90
    Analyser.mosaicv = 90
    Analyser.Q = 1.87278
    Analyser.r0 = 0.8

    ## GENERARING LOWER ANALYZER WEDGE
    # Arm = KVASIR.add_component(f"Armm_{n}", "Arm", AT=[0,0,0], ROTATED=[f"dtS[{n}]",0,0], RELATIVE="Arm_sample")
    # Analyser = KVASIR.add_component(f"Analyzerm_{n}","Monochromator_flat", AT = [0,0,f"len[{n}]"], ROTATED=[f"-dtA_{n}/2",90,0], RELATIVE=f"Armm_{n}" )
    # Analyser.zwidth = f"2*len[{n}]*sin(tDy/2*DEG2RAD)"
    # Analyser.yheight = width
    # Analyser.mosaich = 90
    # Analyser.mosaicv = 90
    # Analyser.Q = 1.87278
    # Analyser.r0 = 0.8

In [7]:
N_detectors  = 1#24#230                                                                         #Number of detectors, centered around wedge (24 is recomended for single wedge to capture mosaic spread)

for n in range(N_detectors):
    ## UPPER ToF MONITOR SECTION
    KVASIR.add_component(f"arm_detector_{n}", "Arm", AT = [0,0,0], RELATIVE="Arm_sample", ROTATED=[0,f"tDy*{n}-tDy*{N_detectors/2}",0])
    det_He3 = KVASIR.add_component(f"det_He3_{n}_ToF", "Monitor_nD", AT = [0,"d_SDy", d_SDz], AT_RELATIVE="PREVIOUS", ROTATED=[0,f"tDy*{n}-tDy*{N_detectors/2}",0], ROTATED_RELATIVE="Arm_sample")
    det_He3.options = '"cylinder, y, limits=[-0.2,0.2], bins=80, t, limits = [0.26,0.28], bins=500"' # t, limits = [0.0069,0.00695] energy, limits = [1.82,1.84]
    det_He3.xwidth = width
    det_He3.yheight = "h_D"
    det_He3.filename = f'"Detector_{n}_ToF.dat"'

    ## UPPER ENERGY MONITOR SECTION
    # det_He3 = KVASIR.add_component(f"det_He3_{n}", "Monitor_nD", AT = [0,0, "-u"], AT_RELATIVE="PREVIOUS")
    # det_He3.options = '"cylinder, y, limits=[-0.2,0.2], bins=80, energy, limits = [1.9,2.1], bins=500"' # t, limits = [0.0069,0.00695] energy, limits = [1.82,1.84]
    # det_He3.xwidth = width
    # det_He3.yheight = "h_D"
    # det_He3.filename = f'"Detector_{n}.dat"'
    # # det_He3.set_WHEN("wedge == 1")



    ## LOWER ToF MONITOR SECTION
    # KVASIR.add_component(f"arm_detector_{n}_lower", "Arm", AT = [0,0,0], RELATIVE="Arm_sample", ROTATED=[0,f"tDy*{n}",0])
    # det_He3 = KVASIR.add_component(f"det_He3_{n}_lower", "Monitor_nD", AT = [0,"-d_SDy", d_SDz], AT_RELATIVE="PREVIOUS", ROTATED=[0,f"tDy*{n}",0], ROTATED_RELATIVE="Arm_sample")
    # det_He3.options = '"cylinder, y, limits=[-0.2,0.2], bins=300, t, limits = [0.27,0.278], bins=500"' # t, limits = [0.0069,0.00695] energy, limits = [1.82,1.84]
    # det_He3.xwidth = width
    # det_He3.yheight = "h_D"
    # det_He3.filename = f'"Detector_{n}_ToF_lower.dat"'

    ## LOWER ENERGY MONITOR SECTION
    # det_He3 = KVASIR.add_component(f"det_He3_{n}_ToF_lower", "Monitor_nD", AT = [0,0, "-u"], AT_RELATIVE="PREVIOUS")
    # det_He3.options = '"cylinder, y, limits=[-0.2,0.2], bins=300, energy, limits = [1.9,2.1], bins=500"' # t, limits = [0.0069,0.00695] energy, limits = [1.82,1.84]
    # det_He3.xwidth = width
    # det_He3.yheight = "h_D"
    # det_He3.filename = f'"Detector_{n}_lower.dat"'


In [8]:
#KVASIR.show_components()

In [9]:
KVASIR.set_parameters(t=50)
KVASIR.settings(ncount=1e6, mpi=6)
data = KVASIR.backengine()


---- Found 5 places in McStas output with keyword 'error'. 

MCPL ERROR: Unable to open file!
----------------------------------------------------------------------
MCPL ERROR: Unable to open file!
----------------------------------------------------------------------
MCPL ERROR: Unable to open file!
----------------------------------------------------------------------
MCPL ERROR: Unable to open file!
----------------------------------------------------------------------
MCPL ERROR: Unable to open file!
--------------------------------------------------------------------------
Primary job  terminated normally, but 1 process returned
a non-zero exit code. Per user-direction, the job has been aborted.
--------------------------------------------------------------------------
--------------------------------------------------------------------------
mpirun detected that one or more processes exited with non-zero status, thus causing
the job to be terminated. The first process to do so wa

In [10]:
ms.make_sub_plot(data, log=1)

No data to plot


In [11]:
#KVASIR.show_instrument(format="window")